# 04 · Agriculture — land-use / crop-land classification (EuroSAT, Sentinel-2)

**CRISP module:** Agriculture · **Task:** classify 64×64 Sentinel-2 patches (640 m) into 10 land-use classes,
including *Annual crop*, *Permanent crop*, *Pasture* and *Herbaceous vegetation* · **Model:** ResNet-18
pretrained on 1 M Sentinel-2 scenes (SSL4EO-S12, MoCo)

Run patch-by-patch over a site, the classifier produces a land-use map, which tells CRISP where the farmland is —
the mask its NDVI crop-health readings should be computed over. Uses 12 bands (B10 dropped) so the model
accepts Sentinel-2 L2A. Official TorchGeo train / val / test splits. Runtime on a T4: about 25 min.

In [ ]:
# ── Setup: run once per Colab session (Runtime ▸ Change runtime type ▸ T4 GPU) ──
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/inayatarshad/trainedmodels_gstp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    DRIVE = Path("/content/drive/MyDrive/gstp_runs")       # checkpoints + results persist here
    REPO = WORK / "trainedmodels_gstp"
    if not REPO.exists():
        if subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)]).returncode != 0:
            # Private repo? Upload the repo zip to MyDrive/trainedmodels_gstp.zip and re-run.
            shutil.unpack_archive("/content/drive/MyDrive/trainedmodels_gstp.zip", WORK)
            REPO = next(WORK.glob("trainedmodels_gstp*"))
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:  # local / Kaggle
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO)
    WORK = DRIVE = REPO / "runs"

sys.path.insert(0, str(REPO / "src"))
DRIVE.mkdir(parents=True, exist_ok=True)

import torch
from gstp.utils import seed_everything, get_device
seed_everything(42)
print("Repo:", REPO, "| device:", get_device(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
MODULE = "agriculture"
from torchgeo.datasets import EuroSAT
from gstp.models import build_s2_classifier, classifier_bands, count_parameters
from gstp import train_cls, viz
from gstp.utils import save_json
import numpy as np, pandas as pd, torch

DATA = WORK / "data" / "eurosat"
RESULTS = REPO / "results" / MODULE
CKPT_DIR = DRIVE / MODULE
for p in (DATA, RESULTS / "figures", CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
# Bands are requested in the channel order of the SSL4EO-S12 pretrained weights
# (B8A after B08). TorchGeo lists B8A last by default, so the order must be explicit.
BANDS = classifier_bands(drop_bands=["B10"])
ds = {sp: EuroSAT(str(DATA), split=sp, bands=BANDS, download=True) for sp in ("train", "val", "test")}
CLASSES = list(ds["train"].classes)
print({k: len(v) for k, v in ds.items()}, "\nclasses:", CLASSES, "\nbands:", BANDS)

In [ ]:
cfg = train_cls.ClsTrainConfig(name="landuse_eurosat", class_names=CLASSES, bands=BANDS, epochs=30, batch_size=128,
                               lr=1e-3, patience=8)
model = build_s2_classifier(len(CLASSES), drop_bands=["B10"], input_size=128)
print(f"{count_parameters(model)/1e6:.1f} M parameters")
summary = train_cls.train(model, train_cls.ScaledDataset(ds["train"], cfg.scale, True),
                          train_cls.ScaledDataset(ds["val"], cfg.scale, False), cfg, CKPT_DIR)
viz.training_curves(summary["history"], metric="val_macro_f1", title="landuse_eurosat",
                    path=RESULTS / "figures" / "curves.png");

## Test once

In [ ]:
rep = train_cls.test_report(model, train_cls.ScaledDataset(ds["test"], cfg.scale, False), cfg, RESULTS / "test_landuse.json")
print(f"Test accuracy {rep['accuracy']:.4f} | macro-F1 {rep['macro_f1']:.4f}")
viz.confusion(rep["confusion_matrix"], CLASSES, "EuroSAT test — row-normalised", RESULTS / "figures" / "confusion.png")
per = pd.DataFrame(rep["per_class"]).T
AGRI_CLASSES = ["AnnualCrop", "PermanentCrop", "Pasture", "HerbaceousVegetation"]
print("Mean F1 over agricultural classes:", per.loc[[c for c in AGRI_CLASSES if c in per.index], "f1"].mean().round(4))
per.to_csv(RESULTS / "per_class.csv")
per.style.format(precision=3)

In [ ]:
from gstp.models import export_onnx
ck = torch.load(CKPT_DIR / "landuse_eurosat_best.pt", weights_only=False)
ck["config"]["extra"] = {"scale": cfg.scale, "patch": 64, "bands": BANDS, "classes": CLASSES, "input_size": 128}
torch.save(ck, CKPT_DIR / "landuse_eurosat_best.pt")
export_onnx(model, len(BANDS), 64, str(CKPT_DIR / "landuse_eurosat.onnx"))
save_json({"summary": {k: v for k, v in summary.items() if k != "history"}, "test": rep}, RESULTS / "summary.json")

In [ ]:
# ── Package results: metrics + figures go to Drive as a zip you can download ──
import shutil
archive = shutil.make_archive(str(DRIVE / f"results_{MODULE}"), "zip", RESULTS)
print("Results zip :", archive)
print("Checkpoints :", sorted(str(p) for p in CKPT_DIR.glob("*.pt")))